# Phase 2: Literature Survey & Model Architecture Design

This notebook covers the literature survey for binary change detection on EO-SAR imagery and the proposed model architecture.

## 1. Literature Survey: Change Detection in Remote Sensing

### 1.1 Traditional Methods

1. **Image Differencing**: Subtract pixel values between pre/post images
2. **Change Vector Analysis (CVA)**: Magnitude and direction of change in feature space
3. **PCA-based methods**: Principal component analysis to detect changes
4. **Post-classification comparison**: Classify each image independently, then compare

### 1.2 Deep Learning Methods

#### Key Architectures for Change Detection:

**1. Siamese Networks**
- Use shared weights for both temporal images
- Extract features independently then compare
- Reference: "Siamese NestedUNet Networks for Change Detection" (Li et al., 2020)

**2. UNet-based Architectures**
- Encoder-decoder structure with skip connections
- SNUNet-CD: Densely connected Siamese + NestedUNet
- MRA-SNet: Multi-Res blocks + Attention Gates

**3. Attention Mechanisms**
- CBAM (Convolutional Block Attention Module)
- Self-attention for global context
- Channel + spatial attention for feature refinement

**4. CNN-Transformer Hybrid**
- CNNs for local features, Transformers for global context
- Reference: MDAF-Net (Multidirectional Attention Fusion Network)

### 1.3 EO-SAR Fusion Approaches

**Multi-modal Fusion Strategies:**

1. **Early Fusion**: ConcatenateEO+SAR channels before backbone
2. **Late Fusion**: Separate encoders, merge at decoder
3. **Intermediate Fusion**: Multi-scale feature concatenation
4. **Attention-based Fusion**: Cross-modality attention

**Key Challenges for EO-SAR:**
- Different physical properties (optical vs microwave)
- SAR speckle noise handling
- Channel-wise normalization differences

### 1.4 Class Imbalance Handling

**Loss Functions for Imbalanced Data:**

1. **Focal Loss** (Lin et al., 2017):
   - Down-weights well-classified examples
   - Focuses on hard examples
   - $FL(p_t) = -\alpha_t(1-p_t)^\gamma \log(p_t)$

2. **Dice Loss**:
   - Directly optimizes for IoU
   - $Dice = 2\frac{|X \cap Y|}{|X| + |Y|}$

3. **Tversky Loss**:
   - Generalized version with separate $\alpha$, $\beta$ weights
   - $TL = 1 - \frac{TP}{TP + \alpha \cdot FN + \beta \cdot FP}$

4. **Combined Loss (Dice + Focal)**:
   - Best of both worlds
   - Used in state-of-the-art methods

## 2. State-of-the-Art Methods (2023-2024)

| Method | Architecture | Key Features | Year |
|--------|-------------|--------------|------|
| SNUNet-CD | Siamese + NestedUNet | Dense skip connections | 2021 |
| MRA-SNet | Siamese + UNet | Multi-Res blocks + Attention Gates | 2021 |
| MDAF-Net | CNN + Transformer | Multidirectional filters + Multi-scale attention | 2024 |
| WBANet | Wavelet + Self-Attention | DWT for downsampling preservation | 2024 |
| SpectroChangeNet | DCT + MRC | Frequency + Spatial domain fusion | 2024 |
| ATCFNet | Lightweight CNN | Cross-level attention for edge devices | 2024 |

## 3. Proposed Architecture: Siamese UNet with Attention

### 3.1 Architecture Overview

Based on our literature survey and the dataset characteristics, we propose:

**Siamese UNet with Dual-Modal Encoder**

- **Encoder**: Two separate encoders for EO (RGB) and SAR (grayscale)
  - Pretrained ResNet34 (ImageNet) for EO branch
  - Modified ResNet34 for SAR branch
- **Fusion**: Concatenate features at each scale level
- **Decoder**: UNet-style with attention gates
- **Output**: Binary change mask (0=No-Change, 1=Change)

### 3.2 Design Rationale

1. **Siamese Architecture**: Standard for change detection tasks
   - Shared weights ensure consistent feature extraction
   - Proven effective in SNUNet-CD, MRA-SNet

2. **Dual-Modal Encoders**: 
   - EO: 3-channel (RGB), SAR: 1-channel
   - Separate encoders prevent modality interference
   - Feature-level fusion at each scale

3. **Attention Gates**:
   - Suppress irrelevant features
   - Highlight changed regions
   - Used successfully in MRA-SNet

4. **Pretrained Backbone**:
   - ResNet34 with ImageNet weights
   - Transfer learning for limited dataset
   - Standard in remote sensing CD tasks

5. **Loss Function**:
   - Focal + Dice combined loss
   - Handles 62:1 class imbalance
   - Focuses on minority change class

### 3.3 Model Specifications

In [ ]:
model_specs = """
PROPOSED MODEL SPECIFICATIONS
==============================

Architecture: Siamese UNet (Dual-Modal)

Encoder:
├── EO Branch (Pre-event): ResNet34 pretrained
│   └── Input: 3 channels (RGB) → 512 feature maps
└── SAR Branch (Post-event): ResNet34 pretrained
    └── Input: 1 channel (grayscale) → 512 feature maps

Fusion: Concatenate at each scale (C64, C128, C256, C512)

Decoder:
├── Up Conv blocks with Attention Gates
├── Skip connections from encoder
└── Output: 1 channel (binary mask)

Input Size: 256×256 (resized from 1024×1024)
Batch Size: 16
Optimizer: AdamW (lr=1e-4, weight_decay=1e-4)
Scheduler: CosineAnnealingLR

Loss: Focal Loss (γ=2.0) + Dice Loss
   - Weight: 0.5 Focal + 0.5 Dice
   - Focus on change class (minority)

Early Stopping: patience=10 (based on Val IoU)
Max Epochs: 50
"""
print(model_specs)

## 4. Alternative Approaches Considered

1. **FC-Siam-Conc**: Simple UNet + Siamese concatenation - too basic
2. **FC-Siam-Diff**: Difference before decoder - loses spatial detail
3. **UNet++ with multiple outputs**: Overcomplicated for our dataset
4. **Pure Transformer**: Not enough data (2781 samples) for ViT
5. **Single encoder (concat inputs)**: Cannot handle EO-SAR modality difference

## 5. Implementation Plan

In [ ]:
implementation_plan = """
IMPLEMENTATION PLAN
====================

Phase 3: Data Pipeline
├── Dataset class (PyTorch Dataset)
├── Label remapping (0,1→0, 2,3→1)
├── Normalization (ImageNet stats for EO, SAR-specific for SAR)
├── Augmentations: Flip, Rotate, Color jitter
└── DataLoader with appropriate batch size

Phase 4: Model Architecture
├── SiameseUNet class
├── ResNet34 encoder (pretrained)
├── Attention Gate module
├── Decoder blocks
└── Forward pass implementation

Phase 5: Training
├── Focal + Dice combined loss
├── AdamW optimizer
├── CosineAnnealing LR scheduler
├── Early stopping (patience=10)
├── Checkpointing (save best based on Val IoU)
└── TensorBoard logging

Phase 6: Evaluation
├── IoU, Precision, Recall, F1
├── Confusion matrix
├── Visualization (success/failure cases)
└── Save predictions
"""
print(implementation_plan)

## 6. References

1. Fang, S., Li, K., Li, Z. (2021). SNUNet-CD: A Densely Connected Siamese Network for Change Detection of VHR Images. IEEE Geoscience and Remote Sensing Letters.

2. Li, K., Li, Z., Fang, S. (2020). Siamese NestedUNet Networks for Change Detection of High Resolution Satellite Image. CCRIS 2020.

3. Chen, P., et al. (2024). Multidirectional Attention Fusion Network for SAR Change Detection. MDPI Remote Sensing.

4. Lin, T.Y., et al. (2017). Focal Loss for Dense Object Detection. ICCV 2017.

5. Chen, H., et al. (2024). Wavelet-based Bi-dimensional Aggregation Network for SAR Image Change Detection. arXiv:2407.13151.

6. Islam, M., et al. (2024). SpectroChangeNet: Spectrotemporal Feature Learning for SAR Change Detection. IEEE TGARS.

---

This concludes Phase 2: Literature Survey & Model Architecture Design.